# U02 · 数学补课 · 教学主文档

**目标**：搞懂神经网络训练需要的 3 个数学工具。

| 章节 | 主题 | 为什么重要 |
|------|------|-----------|
| §1 | 矩阵乘法 | 神经网络的前向传播 = 矩阵乘法 |
| §2 | 偏导数 | 理解「一个参数变一点，loss 变多少」 |
| §3 | 链式法则 | **反向传播的核心**，GRU 反传也靠它 |

> 📌 学习方式：**读一段 → 运行代码 → 自己改数字看结果变化**。不要只看不跑。

---
## §1 矩阵乘法

### 1.1 什么是矩阵
矩阵就是**二维数组**，有「行」和「列」两个维度。一个 $3 \times 4$ 矩阵表示 3 行 4 列。

$$
A = \begin{bmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \end{bmatrix}
$$

### 1.2 维度规则（最最最重要！）

$$
\underbrace{A}_{(m, n)} \times \underbrace{B}_{(n, p)} = \underbrace{C}_{(m, p)}
$$

**口诀**：
- A 的**列数** 必须等于 B 的**行数**（中间的 n 必须相同）
- 结果 C 的形状 = A 的行数 × B 的列数（两头）

### 1.3 计算规则
$C$ 的第 $i$ 行第 $j$ 列元素 = A 的第 $i$ 行 **点乘** B 的第 $j$ 列。

$$ C_{ij} = \sum_{k=1}^{n} A_{ik} \cdot B_{kj} $$

不理解公式没关系，下面代码演示。

In [1]:
import numpy as np

# 一个 2x3 的矩阵
A = np.array([[1, 2, 3],
              [4, 5, 6]])

# 一个 3x2 的矩阵
B = np.array([[1, 0],
              [0, 1],
              [1, 1]])

print('A shape:', A.shape)   # (2, 3)
print('B shape:', B.shape)   # (3, 2)

# 矩阵乘法：@ 符号（或 np.matmul）
C = A @ B
print('C shape:', C.shape)   # (2, 2)
print('C =\n', C)

# 手动验证 C[0,0]
# = A 第 0 行 [1,2,3] 和 B 第 0 列 [1,0,1] 的点积
# = 1*1 + 2*0 + 3*1 = 4
print('手算 C[0,0] =', 1*1 + 2*0 + 3*1)

A shape: (2, 3)
B shape: (3, 2)
C shape: (2, 2)
C =
 [[ 4  5]
 [10 11]]
手算 C[0,0] = 4


### 1.4 维度不匹配会报错

👇 运行下面这段代码，故意制造错误，体会报错信息。

In [2]:
# A 是 (2,3)，如果 B 是 (2,3)，中间维度不匹配
A = np.array([[1, 2, 3], [4, 5, 6]])
B = np.array([[1, 2, 3], [4, 5, 6]])

try:
    C = A @ B
except ValueError as e:
    print('报错:', e)

# 修复方法：把 B 转置（.T）
# B.T 的形状从 (2,3) 变成 (3,2)
C = A @ B.T
print('修复后 C.shape =', C.shape)   # (2,2)

报错: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)
修复后 C.shape = (2, 2)


### 1.5 和神经网络的关系（🎯 核心洞察）

一层全连接神经网络的前向传播就是：

$$ y = x W + b $$

- $x$：输入，形状 `(batch_size, in_dim)`
- $W$：权重矩阵，形状 `(in_dim, out_dim)`
- $y$：输出，形状 `(batch_size, out_dim)`
- $b$：偏置（可广播，形状 `(out_dim,)`）

**💡 为什么用矩阵乘法？**
- 一次处理整个 batch 的所有样本（并行）
- GPU 对矩阵乘法高度优化
- 后面 Embedding、GRU、Attention 全都是矩阵乘法的组合

In [3]:
# 模拟一层神经网络前向
batch_size = 4      # 4 个样本
in_dim = 10         # 输入维度
out_dim = 3         # 输出维度（比如 3 分类）

x = np.random.randn(batch_size, in_dim)      # (4, 10)
W = np.random.randn(in_dim, out_dim)          # (10, 3)
b = np.random.randn(out_dim)                  # (3,)

y = x @ W + b                                  # (4, 3)
print('输入 x:', x.shape)
print('权重 W:', W.shape)
print('输出 y:', y.shape)
print('\n y =\n', y)

输入 x: (4, 10)
权重 W: (10, 3)
输出 y: (4, 3)

 y =
 [[-1.08764712 -0.74159709 -0.84987881]
 [-2.94206073  3.72734102 -4.70057498]
 [ 0.4297597  -3.26803163  2.13748598]
 [-0.57635302 -3.03086342  4.19472801]]


---
## §2 偏导数

### 2.1 先复习单变量导数

导数的直觉：**x 变一点点，y 变多少**。

$$ f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h} $$

常见导数（记住这几个就够了）：

| $f(x)$ | $f'(x)$ |
|--------|---------|
| $x^2$ | $2x$ |
| $x^n$ | $n x^{n-1}$ |
| $e^x$ | $e^x$ |
| $\ln x$ | $1/x$ |
| 常数 $c$ | $0$ |
| $c \cdot f(x)$ | $c \cdot f'(x)$ |

In [4]:
# 用「数值微分」验证 x^2 的导数是 2x
def f(x):
    return x ** 2

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

for x in [1.0, 2.0, 3.0, 5.0]:
    approx = numerical_derivative(f, x)
    true_val = 2 * x
    print(f'x={x}: 数值导数≈{approx:.4f}, 理论值={true_val}')

x=1.0: 数值导数≈2.0000, 理论值=2.0
x=2.0: 数值导数≈4.0000, 理论值=4.0
x=3.0: 数值导数≈6.0000, 理论值=6.0
x=5.0: 数值导数≈10.0000, 理论值=10.0


### 2.2 多变量 → 偏导数

当函数有多个变量，比如 $f(x, y) = x^2 + 3y$：

- **对 x 的偏导** $\frac{\partial f}{\partial x}$：**把 y 当常数**，只对 x 求导 → $2x$
- **对 y 的偏导** $\frac{\partial f}{\partial y}$：**把 x 当常数**，只对 y 求导 → $3$

**一句话记忆：偏导 = 固定其他变量，只动一个**。

**为什么重要？** 神经网络有几百万个参数，loss 是所有参数的函数。训练时要知道：
> 「**只动这一个参数**，loss 会怎么变？」

这就是偏导数 $\frac{\partial L}{\partial W_{ij}}$。

In [5]:
# f(x, y) = x^2 + 3y
def f(x, y):
    return x**2 + 3*y

# 偏导 ∂f/∂x：固定 y，只动 x
def partial_x(f, x, y, h=1e-5):
    return (f(x + h, y) - f(x, y)) / h

# 偏导 ∂f/∂y：固定 x，只动 y
def partial_y(f, x, y, h=1e-5):
    return (f(x, y + h) - f(x, y)) / h

x, y = 2.0, 5.0
print(f'∂f/∂x = {partial_x(f, x, y):.4f}  (理论 2x = {2*x})')
print(f'∂f/∂y = {partial_y(f, x, y):.4f}  (理论 3)')

∂f/∂x = 4.0000  (理论 2x = 4.0)
∂f/∂y = 3.0000  (理论 3)


### 2.3 梯度（Gradient）

把所有偏导数**拼成一个向量**，就叫「梯度」：

$$ \nabla f = \left[ \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right] $$

**梯度方向** = 函数增长最快的方向。

**梯度下降** = 往**负梯度**方向走（下降最快）：
$$ w_{\text{new}} = w_{\text{old}} - \eta \cdot \nabla L $$
其中 $\eta$ 是学习率（步长）。这就是训练神经网络的核心公式。

---
## §3 链式法则 🔥

**这是反向传播的灵魂。把这一节吃透，GRU/LSTM 的反传就不怕了。**

### 3.1 复合函数

当函数是**套娃**的：$y = f(g(x))$

例：$y = (3x + 1)^2$
- 外层 $f(u) = u^2$
- 内层 $u = g(x) = 3x + 1$

### 3.2 链式法则公式

$$ \frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} $$

**直觉**：x 影响 u，u 影响 y，所以 x 对 y 的影响 = 两段影响相乘。

**验证**：
- $\frac{dy}{du} = 2u = 2(3x+1)$
- $\frac{du}{dx} = 3$
- 相乘：$2(3x+1) \cdot 3 = 6(3x+1) = 18x + 6$

（如果直接展开 $y = 9x^2 + 6x + 1$，求导得 $18x + 6$，对上了 ✅）

In [6]:
# 用数值导数验证链式法则
def y(x):
    u = 3*x + 1
    return u ** 2

x = 2.0
# 直接数值求 dy/dx
dy_dx_direct = (y(x + 1e-5) - y(x)) / 1e-5

# 用链式法则算
u = 3*x + 1
dy_du = 2 * u       # 外层导数
du_dx = 3           # 内层导数
dy_dx_chain = dy_du * du_dx

print(f'直接数值微分 dy/dx = {dy_dx_direct:.4f}')
print(f'链式法则    dy/dx = {dy_dx_chain:.4f}')
print(f'理论值 18x+6    = {18*x + 6}')

直接数值微分 dy/dx = 42.0001
链式法则    dy/dx = 42.0000
理论值 18x+6    = 42.0


### 3.3 多层套娃 → 就是「反向传播」

一个 2 层神经网络的前向：

$$
\begin{aligned}
z_1 &= x \cdot w_1 \\
a_1 &= \text{relu}(z_1) \\
z_2 &= a_1 \cdot w_2 \\
L   &= (z_2 - y)^2 \quad \text{(损失)}
\end{aligned}
$$

我们想求 $\frac{\partial L}{\partial w_1}$（loss 对第一层权重的偏导），反向一步步乘回去：

$$
\frac{\partial L}{\partial w_1}
= \frac{\partial L}{\partial z_2} \cdot \frac{\partial z_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_1}
$$

**这就是反向传播！** 每一层的梯度 = 上游梯度 × 本层局部导数。

> 🎯 GRU 的 BPTT（时间反向传播）无非就是在时间维度上套更多层。原理一模一样。

In [7]:
# 手算一个最简化的 2 层网络反向传播
# 前向
x = 2.0
w1 = 0.5
w2 = 1.5
y_true = 3.0

z1 = x * w1              # 1.0
a1 = max(0, z1)          # relu: 1.0
z2 = a1 * w2             # 1.5
L = (z2 - y_true) ** 2   # (1.5 - 3)^2 = 2.25
print(f'前向: z1={z1}, a1={a1}, z2={z2}, L={L}')

# 反向 (链式法则)
dL_dz2 = 2 * (z2 - y_true)       # 2(z2 - y) = -3
dz2_da1 = w2                      # 1.5
da1_dz1 = 1.0 if z1 > 0 else 0.0  # relu 的导数: 正数时=1
dz1_dw1 = x                       # 2.0

dL_dw1 = dL_dz2 * dz2_da1 * da1_dz1 * dz1_dw1
print(f'反向: dL/dw1 = {dL_dw1}')

# 数值微分验证
def forward(w1_val):
    z1 = x * w1_val
    a1 = max(0, z1)
    z2 = a1 * w2
    return (z2 - y_true) ** 2

numerical = (forward(w1 + 1e-5) - forward(w1)) / 1e-5
print(f'数值验证: dL/dw1 ≈ {numerical:.4f}')

前向: z1=1.0, a1=1.0, z2=1.5, L=2.25
反向: dL/dw1 = -9.0
数值验证: dL/dw1 ≈ -8.9999


---
## 🎯 本章完成

如果上面代码你都跑过一遍，并且能回答下面 3 个问题，就可以进入练习：

1. `A.shape=(3,7)`, `B.shape=(?,4)`，中间的 `?` 必须是多少？结果形状多少？
2. $f(x,y,z) = x^2 y + 3z$，$\partial f / \partial y = ?$
3. $y = \sin(3x)$，用链式法则求 $dy/dx$ （提示：$\sin'(u) = \cos(u)$）

👉 自己先想答案，再打开 `exercises.ipynb` 做练习验证。